<a href="https://colab.research.google.com/github/Sanath-cmd/Internship_ITT/blob/main/Algorithms/Fashion_MNIST_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed= 42

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'The device in use {device}')

The device in use cpu


In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')
test_data = pd.read_csv('/content/drive/MyDrive/fashion-mnist_test.csv')

In [ ]:
X_train, y_train = train_data.iloc[:, 1:].values, train_data.iloc[:, 0].values
X_test, y_test = test_data.iloc[:, 1:].values, test_data.iloc[:, 0].values

In [ ]:
X_train = X_train/255
X_test = X_test/255

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype= torch.float32).reshape(-1, 1, 28, 28)
    self.labels = torch.tensor(labels, dtype= torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)


In [ ]:
print(y_train[:20])

[2 9 6 0 3 4 4 5 4 8 0 8 9 0 2 2 9 3 3 3]


In [ ]:
train_loader = DataLoader(train_dataset, batch_size= 128, shuffle = True, pin_memory= True)
test_loader = DataLoader(test_dataset, batch_size= 128, shuffle = False, pin_memory= True)

In [ ]:
class Model(nn.Module):

  def __init__(self, input_size):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels= 1, out_channels= 32, kernel_size= 3)
    self.conv2 = nn.Conv2d(32, 64, 3)

    self.pool = nn.MaxPool2d(2, 2)
    self.fc1 = nn.Linear(64*5*5, 128)
    self.fc2 = nn.Linear(128, 10)

  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = self.pool(x)
    x = F.relu(self.conv2(x))
    x = self.pool(x)
    x = torch.flatten(x, 1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x


In [ ]:
epochs = 100
learning_rate = 0.01

In [ ]:
model = Model(1)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= learning_rate, weight_decay= 1e-4)

In [ ]:
for epoch in range(epochs):
  train_losses = []
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = model(batch_features)
    loss = criterion(output, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss = total_epoch_loss + loss.item()
    train_losses.append(total_epoch_loss/len(train_loader))

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch+1}, Loss: {avg_loss}')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1, Loss: 1.6994536348751612
Epoch: 2, Loss: 0.8398858397754271
Epoch: 3, Loss: 0.7364249853437135
Epoch: 4, Loss: 0.6753915463175092
Epoch: 5, Loss: 0.6260940287031853
Epoch: 6, Loss: 0.5831458588271762
Epoch: 7, Loss: 0.5538116605805435
Epoch: 8, Loss: 0.5280562880705161
Epoch: 9, Loss: 0.5076789758098659
Epoch: 10, Loss: 0.49047142769227914
Epoch: 11, Loss: 0.4756406475740201
Epoch: 12, Loss: 0.4617630031063104
Epoch: 13, Loss: 0.45001283580306245
Epoch: 14, Loss: 0.43862654812046203
Epoch: 15, Loss: 0.4290466191672059
Epoch: 16, Loss: 0.41843002024235754
Epoch: 17, Loss: 0.40961783025056314
Epoch: 18, Loss: 0.40124941981042117
Epoch: 19, Loss: 0.3947469179691282
Epoch: 20, Loss: 0.3878363754068102
Epoch: 21, Loss: 0.3816618837082564
Epoch: 22, Loss: 0.37377229412354385
Epoch: 23, Loss: 0.36930740086127445
Epoch: 24, Loss: 0.3633939710570805
Epoch: 25, Loss: 0.35692032679184665
Epoch: 26, Loss: 0.352436727456955
Epoch: 27, Loss: 0.3479476073530437
Epoch: 28, Loss: 0.3425045063

In [ ]:
model.eval()
total = 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = model(batch_features)
    _ , predicted = torch.max(output.data, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(correct/total)
